# Global Daily Event Analysis: Marine Heatwave ID & Tracking using `MarEx`

This notebook turns the boolean marine-heatwave record from `01_preprocess_extremes.ipynb`
(`extremes_shifting_hobday.zarr`) into tracked events with IDs, areas and centroids, followed through
mergers and splits, with a log of every merger. It writes `events_merged.zarr` and, for comparison, a basic tracking in `events_basic.zarr`. Both feed
`03_visualise_events.ipynb`. The pipeline for gridded data:

1. **Morphological Pre-Processing**
    - Binary closing then opening with a disk of radius `R_fill` cells fills holes and removes features smaller than that
    - Gaps of up to `T_fill` days within an event are bridged in time
    - Objects smaller than `area_filter_absolute` cells are dropped

2. **Blob Identification**
    - Connected components are labelled on each day (8-connected, joined across the dateline)
    - Area and centroid are computed for every object

3. **Temporal Tracking**
    - Objects on consecutive days are linked where they overlap by at least `overlap_threshold` of the smaller area
    - When objects merge, the merged object keeps the parents' IDs, and each cell goes to the parent of the nearest
      cell (`nn_partitioning=True`) or of the nearest centroid

4. **Graph Reduction & Finalisation**
    - The overlap graph is reduced with `scipy.sparse.csgraph.connected_components` to one global ID per event
    - Events are mapped into ID-time space, with area, centroid, start and end for each

Every parameter is described under [Tracking & Merge/Split](https://marex.readthedocs.io/en/latest/guide/tracking.html#parameters).

Compute: one compute node runs a local cluster of 16 workers × 8 threads × 14 GB. The table at the end reports
wall time and memory per stage.

## Setup

In [1]:
from getpass import getuser
from pathlib import Path

import dask
import xarray as xr

import marEx
import marEx.helper as hpc

2026-10-18 22:42:54 - marEx - INFO - [PID:616142] - MarEx logging configured - Level: INFO, Mode: normal


In [2]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()  # Lustre scratch
out_dir = scratch_dir / "marEx" / "examples" / "gridded"  # where 01_preprocess_extremes.ipynb wrote its output
temp_dir = scratch_dir / "marEx" / "examples" / "tmp"  # staging for compute_mode="streaming"

# One compute node: 16 workers x 8 threads x 14 GB.
n_workers, threads_per_worker, memory_limit = 16, 8, "14GB"

In [3]:
out_dir.mkdir(parents=True, exist_ok=True)

# graphs of a few hundred MB are expected at this scale
dask.config.set({"distributed.admin.large-graph-warning-threshold": "1GB"})

# memory_limit is passed explicitly because on an HPC node Dask sizes workers from the whole node's memory,
# not from the job's allocation, and without it the workers can claim more than the job was granted.
client = hpc.start_local_cluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    memory_limit=memory_limit,
    scratch_dir=scratch_dir / "marEx" / "dask",
)

# Records wall time, peak memory, spill and worker restarts for each `monitor.stage(...)` block below
monitor = hpc.ResourceMonitor(client)

2026-10-18 22:43:02 - marEx.helper.cluster - INFO - [PID:616142] - Starting local Dask cluster with 16 workers, 8 threads each


2026-10-18 22:43:02 - marEx.helper.dask_config - INFO - [PID:616142] - Configuring Dask for HPC environment


2026-10-18 22:43:02 - marEx.helper.dask_config - INFO - [PID:616142] - Dask temporary directory: /scratch/<u>/<user>/marEx/dask/tmpdzy_h1el


2026-10-18 22:43:02 - marEx.helper.dask_config - INFO - [PID:616142] - Dask configuration completed


2026-10-18 22:43:02 - marEx.helper.cluster - INFO - [PID:616142] - System resources: 128 physical cores, 256 logical cores, 251.6GB total memory


...

2026-10-18 22:43:02 - marEx.helper.cluster - INFO - [PID:616142] - Starting Local cluster startup


2026-10-18 22:43:05 - marEx.helper.cluster - INFO - [PID:616142] - Completed Local cluster startup in 2.34s


Hostname: l30287
Forward Port: l30287:8787
Dashboard Link: localhost:8787/status
2026-10-18 22:43:05 - marEx.helper.cluster - INFO - [PID:616142] - Local cluster started successfully - Dashboard: http://127.0.0.1:8787/status


2026-10-18 22:43:05 - marEx.helper.cluster - INFO - [PID:616142] - After cluster startup - Memory Usage - RSS: 573.0MB, Virtual: 27292.3MB, Percent: 0.2%, Available: 244429.4MB


2026-10-18 22:43:05 - marEx.helper.resources - INFO - [PID:616142] - Cluster: 16 workers, 128 threads, memory_limit per worker [14.0] GB (224 GB total)


## 1. Tracking with Merging & Splitting

At 25 years of daily data one int32 field is 9282 × 720 × 1440 × 4 B ≈ 38 GB, and `compute_mode="persist"` keeps several
such fields in cluster memory at once. `compute_mode="streaming"` stages them to zarr under `temp_dir` instead, which cuts the
memory pinned in the cluster. The per-object properties and the overlap graph stay in memory and grow with the
number of events. The staging directory outlives `run()`, so the result
is written first and the directory then released with `marEx.clear_staging`. See
[Memory Parameters](https://marex.readthedocs.io/en/latest/guide/tracking.html#compute-modes-and-larger-than-memory-tracking).

In [4]:
# Space stays whole because the morphology and the labelling are global in space. Time is the axis to chunk.
ds = xr.open_zarr(out_dir / "extremes_shifting_hobday.zarr", chunks={"time": 25, "lat": -1, "lon": -1})
# The lat/lon grid's anisotropy near the pole biases identification & tracking, so cells north of 85° are masked out
mask = ds.mask.where((ds.lat < 85) & (ds.lat > -90), other=False)
ds

<xarray.Dataset> Size: 50GB
Dimensions:         (time: 9282, lat: 720, lon: 1440, dayofyear: 366)
Coordinates:
  * dayofyear       (dayofyear) uint16 732B 1 2 3 4 5 6 ... 362 363 364 365 366
  * lat             (lat) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * lon             (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.4 179.6 179.9
  * time            (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-0...
Data variables:
    dat_anomaly     (time, lat, lon) float32 38GB dask.array<chunksize=(25, 720, 1440), meta=np.ndarray>
    extreme_events  (time, lat, lon) bool 10GB dask.array<chunksize=(25, 720, 1440), meta=np.ndarray>
    mask            (lat, lon) bool 1MB dask.array<chunksize=(720, 1440), meta=np.ndarray>
    thresholds      (lat, lon, dayofyear) float32 2GB dask.array<chunksize=(720, 1440, 25), meta=np.ndarray>
Attributes: (12/13)
    max_anomaly:           21.038787841796875
    method_anomaly:        shifting_baseline
    method_extreme:        seasonal_percentile
    method_percentile:     approximate
    precision:             0.014025858561197917
    preprocessing_steps:   ['Rolling climatology using 15 years', 'Smoothed w...
    ...                    ...
    smooth_max_missing:    0.1
    tail:                  upper
    threshold_percentile:  95
    window_days:           11
    window_spatial:        5
    window_years:          15

In [5]:
with monitor.stage("Track: merging & splitting"):
    tracker = marEx.tracker(
        ds.extreme_events,
        mask,
        grid_resolution=0.25,  # degrees, for object areas
        area_filter_absolute=600,  # drop objects smaller than 600 cells
        R_fill=12,  # fill holes up to 12 cells in radius
        T_fill=4,  # bridge gaps of up to 4 days within one event
        allow_merging=True,  # follow events through mergers and splits, recording the mergers in `merges_ds`
        overlap_threshold=0.25,  # link objects that overlap by at least 25 % of the smaller area
        nn_partitioning=True,  # partition merged objects by nearest parent cell (False: nearest centroid, Sun et al. 2023)
        compute_mode="streaming",
        temp_dir=str(temp_dir),
    )
    extreme_events_ds, merges_ds = tracker.run(return_merges=True)
    extreme_events_ds.to_zarr(out_dir / "events_merged.zarr", mode="w")
    merges_ds.to_netcdf(out_dir / "events_merged_merges.nc", mode="w")

marEx.clear_staging(extreme_events_ds)
extreme_events_ds = xr.open_zarr(out_dir / "events_merged.zarr", chunks={})
extreme_events_ds

2026-10-18 22:43:05 - marEx.track.tracker - INFO - [PID:616142] - Initialising MarEx tracker


2026-10-18 22:43:05 - marEx.track.tracker - INFO - [PID:616142] - Grid type: structured


2026-10-18 22:43:05 - marEx.track.tracker - INFO - [PID:616142] - Parameters: R_fill=12, T_fill=4, area_filter_quartile=None, area_filter_absolute=600, prefilter_min_cells=None


2026-10-18 22:43:07 - marEx.core.compute_mode - INFO - [PID:616142] - Streaming staging directory: /scratch/<u>/<user>/marEx/examples/tmp/marex_stage_616142_b3db9d9e


2026-10-18 22:43:10 - marEx.track.tracker - INFO - [PID:616142] - Calculating cell areas from grid resolution: 0.25 degrees


...

Tracking Statistics:
   Binary Hobday to Processed Area Fraction: 0.3860165838356871
   Total Object Area IDed (cells): 751319940.0
   Number of Initial Pre-Filtered Objects: 173366
   Number of Final Filtered Objects: 131138
   Area Cutoff Threshold (cells): 600
   Accepted Area Fraction: 0.9865351863282106
   Total Events Tracked: 9295
   Total Merging Events Recorded: 37289


2026-10-18 23:22:12 - marEx.track.tracker - INFO - [PID:616142] - Completed Computing event statistics and attributes in 3.32s


2026-10-18 23:22:12 - marEx.track.tracker - INFO - [PID:616142] - Tracking pipeline completed successfully - 9295 events identified


Track: merging & splitting: 41.2 min, peak worker memory 69.5 GB of 224 GB (largest worker 6.5 of 14.0 GB), client 1.9 GB, spilled 0.0 GB, restarts 0


<xarray.Dataset> Size: 45GB
Dimensions:       (ID: 9295, time: 9282, lat: 720, lon: 1440, component: 2,
                   sibling_ID: 15)
Coordinates:
  * ID            (ID) int32 37kB 1 2 3 4 5 6 ... 9290 9291 9292 9293 9294 9295
  * component     (component) int64 16B 0 1
  * lat           (lat) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * lon           (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.4 179.6 179.9
  * time          (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-05-...
Dimensions without coordinates: sibling_ID
Data variables:
    ID_field      (time, lat, lon) int32 38GB dask.array<chunksize=(1, 720, 1440), meta=np.ndarray>
    area          (time, ID) float32 345MB dask.array<chunksize=(1, 9295), meta=np.ndarray>
    centroid      (component, time, ID) float32 690MB dask.array<chunksize=(1, 1, 9295), meta=np.ndarray>
    global_ID     (time, ID) int32 345MB dask.array<chunksize=(1, 9295), meta=np.ndarray>
    merge_ledger  (time, ID, sibling_ID) int32 5GB dask.array<chunksize=(1, 9295, 15), meta=np.ndarray>
    presence      (time, ID) bool 86MB dask.array<chunksize=(1, 9295), meta=np.ndarray>
    time_end      (ID) datetime64[ns] 74kB dask.array<chunksize=(9295,), meta=np.ndarray>
    time_start    (ID) datetime64[ns] 74kB dask.array<chunksize=(9295,), meta=np.ndarray>
Attributes: (12/14)
    N_events_final:              9295
    N_objects_filtered:          131138
    N_objects_prefiltered:       173366
    R_fill:                      12
    T_fill:                      4
    accepted_area_fraction:      0.9865351863282106
    ...                          ...
    area_threshold (cells):      600
    multi_parent_merges:         13804
    nn_partitioning:             1
    overlap_threshold:           0.25
    preprocessed_area_fraction:  0.3860165838356871
    total_merges:                37289

In [6]:
merges_ds

<xarray.Dataset> Size: 7MB
Dimensions:        (merge_ID: 37289, parent_idx: 15, child_idx: 15)
Dimensions without coordinates: merge_ID, parent_idx, child_idx
Data variables:
    parent_IDs     (merge_ID, parent_idx) int32 2MB 59 61 -1 -1 ... -1 -1 -1 -1
    child_IDs      (merge_ID, child_idx) int32 2MB 73 131139 -1 -1 ... -1 -1 -1
    overlap_areas  (merge_ID, parent_idx) int32 2MB 6172 1645 -1 -1 ... -1 -1 -1
    merge_time     (merge_ID) datetime64[ns] 298kB 1997-01-06T12:00:00 ... 20...
    n_parents      (merge_ID) int8 37kB 2 2 2 2 2 2 2 2 2 ... 2 2 2 2 2 2 2 2 2
    n_children     (merge_ID) int8 37kB 2 2 2 2 2 2 2 2 2 ... 2 2 2 2 2 2 2 2 2
Attributes:
    fill_value:  -1

## 2. Basic Tracking for Comparison

The standard method in the literature (e.g. Scannell et al.): no temporal gap filling, no merging or splitting, and no
independent tracking of the parts of a merged event.

In [7]:
with monitor.stage("Track: basic"):
    basic_tracker = marEx.tracker(
        # 100-day time chunks (four of the stored 25-day chunks): without merging, events are labelled in 3-D, and the
        # labelling's task graph grows with the number of time chunks, so fewer, longer chunks keep it small
        ds.extreme_events.chunk({"time": 100}),
        mask,
        area_filter_absolute=600,
        R_fill=12,
        T_fill=0,  # no temporal gap filling
        allow_merging=False,  # touching objects share one ID for all time, before and after they touch
        compute_mode="streaming",
        temp_dir=str(temp_dir),
    )
    extreme_events_basic_ds = basic_tracker.run()
    extreme_events_basic_ds.to_zarr(out_dir / "events_basic.zarr", mode="w")

marEx.clear_staging(extreme_events_basic_ds)
extreme_events_basic_ds = xr.open_zarr(out_dir / "events_basic.zarr", chunks={})
extreme_events_basic_ds

2026-10-18 23:24:17 - marEx.track.tracker - INFO - [PID:616142] - Initialising MarEx tracker


2026-10-18 23:24:17 - marEx.track.tracker - INFO - [PID:616142] - Grid type: structured


2026-10-18 23:24:17 - marEx.track.tracker - INFO - [PID:616142] - Parameters: R_fill=12, T_fill=0, area_filter_quartile=None, area_filter_absolute=600, prefilter_min_cells=None


2026-10-18 23:24:17 - marEx.core.compute_mode - INFO - [PID:616142] - Streaming staging directory: /scratch/<u>/<user>/marEx/examples/tmp/marex_stage_616142_9bd1a8e4


2026-10-18 23:24:17 - marEx.track.tracker - INFO - [PID:616142] - No cell_areas provided for structured grid - using unit areas (cell counts)


...

Tracking Statistics:
   Binary Hobday to Processed Area Fraction: 0.4506464101542352
   Total Object Area IDed (cells): 645331428.0
   Number of Initial Pre-Filtered Objects: 186118
   Number of Final Filtered Objects: 135897
   Area Cutoff Threshold (cells): 600
   Accepted Area Fraction: 0.9838405669590293
   Total Events Tracked: 13176


2026-10-18 23:25:40 - marEx.track.tracker - INFO - [PID:616142] - Completed Computing event statistics and attributes in 1.13s


2026-10-18 23:25:40 - marEx.track.tracker - INFO - [PID:616142] - Tracking pipeline completed successfully - 13176 events identified


Track: basic: 1.8 min, peak worker memory 68.4 GB of 224 GB (largest worker 4.7 of 14.0 GB), client 1.7 GB, spilled 0.0 GB, restarts 0


<xarray.Dataset> Size: 38GB
Dimensions:   (time: 9282, lat: 720, lon: 1440)
Coordinates:
  * lat       (lat) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * lon       (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.4 179.6 179.9
  * time      (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-05-31T1...
Data variables:
    ID_field  (time, lat, lon) int32 38GB dask.array<chunksize=(1, 720, 1440), meta=np.ndarray>
Attributes:
    N_events_final:              13176
    N_objects_filtered:          135897
    N_objects_prefiltered:       186118
    R_fill:                      12
    T_fill:                      0
    accepted_area_fraction:      0.9838405669590293
    allow_merging:               0
    area_filter_quartile:        0.0
    area_threshold (cells):      600
    preprocessed_area_fraction:  0.4506464101542352

## Wall Time and Memory

Peak memory summed over workers, the largest single worker, and the client (this kernel), per stage.

In [8]:
monitor.summary()

,wall (min),peak worker memory (GB),peak per worker (GB),peak client memory (GB),peak spilled to disk (GB),worker restarts
stage,,,,,,
Track: merging & splitting,41.17,69.45,6.47,1.89,0.0,0
Track: basic,1.80,68.40,4.74,1.70,0.0,0
total,42.96,69.45,6.47,1.89,0.0,0


In [9]:
client.close()